In [1]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta


np.random.seed(42)
random.seed(42)

In [2]:


NUM_PRODUCTS = 50
START_DATE = datetime(2023, 1, 1)
END_DATE = datetime(2024, 12, 31)

CATEGORIES = ['Electronics', 'Grocery', 'Clothing', 'Home & Kitchen', 'Toys']
WAREHOUSES = ['WH_North', 'WH_South', 'WH_East', 'WH_West']
SUPPLIERS = ['Supplier_A', 'Supplier_B', 'Supplier_C', 'Supplier_D']
SEASONS = {12: 'Winter', 1: 'Winter', 2: 'Winter',
           3: 'Spring', 4: 'Spring', 5: 'Spring',
           6: 'Summer', 7: 'Summer', 8: 'Summer',
           9: 'Autumn', 10: 'Autumn', 11: 'Autumn'}

print(f"We will generate data for {NUM_PRODUCTS} products")
print(f"From {START_DATE.date()} to {END_DATE.date()}")

We will generate data for 50 products
From 2023-01-01 to 2024-12-31


In [3]:

products = []

for i in range(1, NUM_PRODUCTS + 1):
    product_id = f"P{i:03d}"  # formats as P001, P002, ... P050
    category = random.choice(CATEGORIES)
    warehouse = random.choice(WAREHOUSES)
    supplier = random.choice(SUPPLIERS)
    
 
    lead_time = random.choice([3, 5, 7, 10, 15])
 
    if category == 'Electronics':
        price = round(random.uniform(2000, 50000), 2)
    elif category == 'Clothing':
        price = round(random.uniform(300, 3000), 2)
    elif category == 'Grocery':
        price = round(random.uniform(20, 500), 2)
    else:
        price = round(random.uniform(200, 5000), 2)
    
    
    base_demand = random.randint(5, 50)
    
    products.append({
        'Product': product_id,
        'Category': category,
        'Warehouse': warehouse,
        'Supplier': supplier,
        'Lead_Time': lead_time,
        'Price': price,
        'Base_Demand': base_demand
    })

products_df = pd.DataFrame(products)
products_df.head()

,Product,Category,Warehouse,Supplier,Lead_Time,Price,Base_Demand
0,P001,Electronics,WH_North,Supplier_C,5,12714.12,11
1,P002,Toys,WH_North,Supplier_D,3,343.03,18
2,P003,Grocery,WH_North,Supplier_B,15,221.37,33
3,P004,Toys,WH_East,Supplier_A,5,3551.07,26
4,P005,Clothing,WH_South,Supplier_B,7,575.97,29


In [4]:


date_range = pd.date_range(start=START_DATE, end=END_DATE, freq='D')

records = []

for _, product in products_df.iterrows():
    current_inventory = product['Base_Demand'] * random.randint(5, 15)
    
    for date in date_range:
        month = date.month
        season = SEASONS[month]
        promotion = np.random.choice([1, 0], p=[0.08, 0.92])
        
        seasonal_multiplier = 1.0
        if product['Category'] == 'Toys' and season == 'Winter':
            seasonal_multiplier = 1.8
        elif product['Category'] == 'Clothing' and season in ['Winter', 'Summer']:
            seasonal_multiplier = 1.4
        elif product['Category'] == 'Electronics' and season == 'Winter':
            seasonal_multiplier = 1.3
        
        promo_multiplier = 1.6 if promotion == 1 else 1.0
        
        expected_demand = product['Base_Demand'] * seasonal_multiplier * promo_multiplier
        raw_demand = max(0, int(np.random.normal(loc=expected_demand, scale=expected_demand * 0.2)))
        
        
        
        records.append({
            'Product': product['Product'],
            'Category': product['Category'],
            'Warehouse': product['Warehouse'],
            'Supplier': product['Supplier'],
            'Date': date,
            'Season': season,
            'Promotion': promotion,
            'Price': product['Price'],
            'Lead_Time': product['Lead_Time'],
            'Raw_Demand': raw_demand,         
            'Historical_Sales': None,         
            'Returns': None,              
            'Current_Inventory': None         
        })

sales_df = pd.DataFrame(records)
print(f"Total rows generated: {len(sales_df)}")
sales_df.head()

Total rows generated: 36550


,Product,Category,Warehouse,Supplier,Date,Season,Promotion,Price,Lead_Time,Raw_Demand,Historical_Sales,Returns,Current_Inventory
0,P001,Electronics,WH_North,Supplier_C,2023-01-01,Winter,0,12714.12,5,11,None,None,None
1,P001,Electronics,WH_North,Supplier_C,2023-01-02,Winter,0,12714.12,5,15,None,None,None
2,P001,Electronics,WH_North,Supplier_C,2023-01-03,Winter,1,12714.12,5,24,None,None,None
3,P001,Electronics,WH_North,Supplier_C,2023-01-04,Winter,0,12714.12,5,17,None,None,None
4,P001,Electronics,WH_North,Supplier_C,2023-01-05,Winter,1,12714.12,5,20,None,None,None


In [5]:


REORDER_MULTIPLIER = 1.5
final_records = []

for product_id in sales_df['Product'].unique():
    product_data = sales_df[sales_df['Product'] == product_id].sort_values('Date').reset_index(drop=True)
    base_demand = products_df[products_df['Product'] == product_id]['Base_Demand'].values[0]
    lead_time = product_data['Lead_Time'].iloc[0]
    
    reorder_point = base_demand * REORDER_MULTIPLIER
    reorder_qty = base_demand * 10
    
    pending_orders = []
    current_inventory = base_demand * random.randint(5, 15)  # starting stock, same logic as before
    
    for idx in range(len(product_data)):
       
        arrived_qty = sum(qty for arrival_idx, qty in pending_orders if arrival_idx == idx)
        if arrived_qty > 0:
            current_inventory += arrived_qty
            pending_orders = [(a, q) for a, q in pending_orders if a != idx]
        
      
        raw_demand = product_data.loc[idx, 'Raw_Demand']
        actual_sales = min(raw_demand, current_inventory)
        
      
        returns = int(actual_sales * random.uniform(0, 0.05))
        

        current_inventory = current_inventory - actual_sales + returns
   
        if current_inventory < reorder_point and len(pending_orders) == 0:
            arrival_idx = idx + lead_time
            pending_orders.append((arrival_idx, reorder_qty))
        
       
        product_data.loc[idx, 'Historical_Sales'] = actual_sales
        product_data.loc[idx, 'Returns'] = returns
        product_data.loc[idx, 'Current_Inventory'] = current_inventory
    
    final_records.append(product_data)

sales_df = pd.concat(final_records, ignore_index=True)


sales_df['Historical_Sales'] = sales_df['Historical_Sales'].astype(int)
sales_df['Returns'] = sales_df['Returns'].astype(int)
sales_df['Current_Inventory'] = sales_df['Current_Inventory'].astype(int)

print("Restocking + sales logic applied correctly.")
sales_df[sales_df['Product'] == 'P001'][['Date','Raw_Demand','Historical_Sales','Current_Inventory']].head(20)

Restocking + sales logic applied correctly.


,Date,Raw_Demand,Historical_Sales,Current_Inventory
0,2023-01-01,11,11,99
1,2023-01-02,15,15,84
2,2023-01-03,24,24,60
3,2023-01-04,17,17,43
4,2023-01-05,20,20,23
5,2023-01-06,12,12,11
6,2023-01-07,6,6,5
7,2023-01-08,17,5,0
8,2023-01-09,13,0,0
9,2023-01-10,12,0,0


In [6]:


sales_df['Lost_Sales'] = sales_df['Raw_Demand'] - sales_df['Historical_Sales']
sales_df['Stockout_Flag'] = (sales_df['Current_Inventory'] == 0).astype(int)

sales_df.head()
print(f"Total stockout-days across dataset: {sales_df['Stockout_Flag'].sum()}")
print(f"Total lost sales (units): {sales_df['Lost_Sales'].sum()}")

Total stockout-days across dataset: 14043
Total lost sales (units): 394684


In [7]:


output_path = '../data/inventory_sales_data.csv'
sales_df.to_csv(output_path, index=False)

print(f"Saved {len(sales_df)} rows to {output_path}")
sales_df.info()

Saved 36550 rows to ../data/inventory_sales_data.csv
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36550 entries, 0 to 36549
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Product            36550 non-null  object        
 1   Category           36550 non-null  object        
 2   Warehouse          36550 non-null  object        
 3   Supplier           36550 non-null  object        
 4   Date               36550 non-null  datetime64[ns]
 5   Season             36550 non-null  object        
 6   Promotion          36550 non-null  int64         
 7   Price              36550 non-null  float64       
 8   Lead_Time          36550 non-null  int64         
 9   Raw_Demand         36550 non-null  int64         
 10  Historical_Sales   36550 non-null  int64         
 11  Returns            36550 non-null  int64         
 12  Current_Inventory  36550 non-null  int64         
 13  Lost_Sal